# ResNet — Deep Residual Learning for Image Recognition

**Paper:** He, K., Zhang, X., Ren, S., & Sun, J. (2016). Deep Residual Learning for Image Recognition. *CVPR 2016*. https://arxiv.org/abs/1512.03385

**Implementation:** `hale_core/nn/vision/cnn/resnet.py`

> ResNet introduces shortcut (skip) connections that allow gradients to flow directly through many layers, solving the degradation problem and enabling training of networks with 100–1000+ layers.

## Architecture Overview

```
Input [B, 3, 224, 224]
       │
  ┌────▼────────────────────────────┐
  │  STEM                           │
  │  7×7 conv, stride=2 → BN→ReLU  │  → [B,  64, 112, 112]
  │  MaxPool, stride=2              │  → [B,  64,  56,  56]
  └─────────────────────────────────┘
       │
  ┌────▼──────────────────────────────────────┐
  │  STAGE 0  (stride=1, channels=64/256)     │
  │  [BasicBlock×2] or [Bottleneck×3]         │  → [B,  64/256,  56, 56]
  └───────────────────────────────────────────┘
       │
  ┌────▼──────────────────────────────────────┐
  │  STAGE 1  (stride=2, channels=128/512)    │
  │  [BasicBlock×2/4] or [Bottleneck×4]       │  → [B, 128/512,  28, 28]
  └───────────────────────────────────────────┘
       │
  ┌────▼──────────────────────────────────────┐
  │  STAGE 2  (stride=2, channels=256/1024)   │
  │  [BasicBlock×2/6] or [Bottleneck×6/23]    │  → [B, 256/1024, 14, 14]
  └───────────────────────────────────────────┘
       │
  ┌────▼──────────────────────────────────────┐
  │  STAGE 3  (stride=2, channels=512/2048)   │
  │  [BasicBlock×2/3] or [Bottleneck×3/8]     │  → [B, 512/2048,  7,  7]
  └───────────────────────────────────────────┘
       │
  ┌────▼──────────────────────────────────┐
  │  GlobalAvgPool → Flatten → Dropout   │  → [B, embed_dim]
  └───────────────────────────────────────┘

BasicBlock (ResNet-18/34):    Bottleneck (ResNet-50/101/152/200):
  x → 3×3 conv → BN → ReLU     x → 1×1 conv → BN → ReLU
    → 3×3 conv → BN               → 3×3 conv → BN → ReLU
    → (+x via skip)               → 1×1 conv → BN
    → ReLU                        → (+x via skip)
                                  → ReLU
```

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch
import numpy as np

fig, axes = plt.subplots(1, 2, figsize=(16, 10))
fig.patch.set_facecolor('#0d1117')
for ax in axes:
    ax.set_facecolor('#0d1117')
    ax.set_xlim(0, 10)
    ax.set_ylim(0, 12)
    ax.axis('off')

# ── Helper ────────────────────────────────────────────────────────────────────
def box(ax, x, y, w, h, label, sublabel='', color='#1f6feb', text_color='white', fontsize=9):
    rect = FancyBboxPatch((x, y), w, h, boxstyle='round,pad=0.1',
                           facecolor=color, edgecolor='#58a6ff', linewidth=1.5)
    ax.add_patch(rect)
    ax.text(x + w/2, y + h/2 + (0.12 if sublabel else 0), label,
            ha='center', va='center', color=text_color,
            fontsize=fontsize, fontweight='bold', fontfamily='monospace')
    if sublabel:
        ax.text(x + w/2, y + h/2 - 0.18, sublabel,
                ha='center', va='center', color='#8b949e', fontsize=7)

def arrow(ax, x1, y1, x2, y2, color='#58a6ff'):
    ax.annotate('', xy=(x2, y2), xytext=(x1, y1),
                arrowprops=dict(arrowstyle='->', color=color, lw=1.5))

# ══════════════════════════════════════════════════════════
# LEFT: BasicBlock (ResNet-18/34)
# ══════════════════════════════════════════════════════════
ax = axes[0]
ax.text(5, 11.5, 'BasicBlock (ResNet-18/34)', ha='center', color='#f0f6fc',
        fontsize=11, fontweight='bold')

# input
box(ax, 3.5, 10.5, 3, 0.6, 'Input x', '[B, C, H, W]', color='#21262d')
# 3x3 conv
box(ax, 3.5, 9.2, 3, 0.7, '3×3 Conv', 'stride, padding=1', color='#1f6feb')
arrow(ax, 5, 10.5, 5, 9.9)
# BN+ReLU
box(ax, 3.5, 8.1, 3, 0.7, 'BN → ReLU', '', color='#388bfd')
arrow(ax, 5, 9.2, 5, 8.8)
# 3x3 conv 2
box(ax, 3.5, 6.9, 3, 0.7, '3×3 Conv', 'stride=1, padding=1', color='#1f6feb')
arrow(ax, 5, 8.1, 5, 7.6)
# BN (no relu yet)
box(ax, 3.5, 5.8, 3, 0.7, 'BatchNorm', '(no ReLU yet)', color='#388bfd')
arrow(ax, 5, 6.9, 5, 6.5)
# skip connection
box(ax, 0.5, 7.5, 2, 0.6, '1×1 Conv+BN', 'projection shortcut\n(if dims differ)', color='#da3633', fontsize=8)
ax.annotate('', xy=(2.5, 7.8), xytext=(5, 10.5),
            arrowprops=dict(arrowstyle='->', color='#f85149', lw=1.5,
                            connectionstyle='arc3,rad=0.3'))
# add
box(ax, 3.5, 4.7, 3, 0.7, 'Add (+)', 'residual + skip', color='#2ea043')
arrow(ax, 5, 5.8, 5, 5.4)
ax.annotate('', xy=(4.5, 5.0), xytext=(2.5, 7.5),
            arrowprops=dict(arrowstyle='->', color='#f85149', lw=1.5))
# ReLU
box(ax, 3.5, 3.6, 3, 0.7, 'ReLU', '', color='#388bfd')
arrow(ax, 5, 4.7, 5, 4.3)
# output
box(ax, 3.5, 2.5, 3, 0.7, 'Output', '[B, planes, H\', W\']', color='#21262d')
arrow(ax, 5, 3.6, 5, 3.2)

ax.text(5, 2.0, 'expansion=1  (planes==out)', ha='center', color='#8b949e', fontsize=8)
ax.text(5, 1.5, 'Used in: ResNet-18 [2,2,2,2]  ResNet-34 [3,4,6,3]', ha='center', color='#8b949e', fontsize=8)

# ══════════════════════════════════════════════════════════
# RIGHT: Bottleneck (ResNet-50/101/152/200)
# ══════════════════════════════════════════════════════════
ax = axes[1]
ax.text(5, 11.5, 'Bottleneck (ResNet-50/101/152/200)', ha='center',
        color='#f0f6fc', fontsize=11, fontweight='bold')

box(ax, 3.5, 10.5, 3, 0.6, 'Input x', '[B, C, H, W]', color='#21262d')
# 1x1 squeeze
box(ax, 3.5, 9.3, 3, 0.7, '1×1 Conv', 'squeeze: C → planes', color='#1f6feb')
arrow(ax, 5, 10.5, 5, 10.0)
box(ax, 3.5, 8.2, 3, 0.7, 'BN → ReLU', '', color='#388bfd')
arrow(ax, 5, 9.3, 5, 8.9)
# 3x3
box(ax, 3.5, 7.1, 3, 0.7, '3×3 Conv', 'stride, padding=1', color='#1f6feb')
arrow(ax, 5, 8.2, 5, 7.8)
box(ax, 3.5, 6.0, 3, 0.7, 'BN → ReLU', '', color='#388bfd')
arrow(ax, 5, 7.1, 5, 6.7)
# 1x1 expand
box(ax, 3.5, 4.9, 3, 0.7, '1×1 Conv', 'expand: planes → planes×4', color='#1f6feb')
arrow(ax, 5, 6.0, 5, 5.6)
box(ax, 3.5, 3.8, 3, 0.7, 'BatchNorm', '(no ReLU before add)', color='#388bfd')
arrow(ax, 5, 4.9, 5, 4.5)
# skip
box(ax, 0.2, 7.0, 2.2, 0.6, '1×1 Conv+BN', 'C → planes×4', color='#da3633', fontsize=8)
ax.annotate('', xy=(2.4, 7.3), xytext=(5, 10.5),
            arrowprops=dict(arrowstyle='->', color='#f85149', lw=1.5,
                            connectionstyle='arc3,rad=0.35'))
# add
box(ax, 3.5, 2.7, 3, 0.7, 'Add (+)', 'residual + skip', color='#2ea043')
arrow(ax, 5, 3.8, 5, 3.4)
ax.annotate('', xy=(4.5, 3.0), xytext=(2.4, 7.0),
            arrowprops=dict(arrowstyle='->', color='#f85149', lw=1.5))
# relu
box(ax, 3.5, 1.6, 3, 0.7, 'ReLU → Output', '[B, planes×4, H\', W\']', color='#21262d')
arrow(ax, 5, 2.7, 5, 2.3)

ax.text(5, 1.1, 'expansion=4  (out = planes × 4)', ha='center', color='#8b949e', fontsize=8)
ax.text(5, 0.6, 'Last BN weight zero-init'd (Goyal et al. 2017 — near-identity start)', ha='center', color='#8b949e', fontsize=7)

plt.suptitle('ResNet Block Architectures', fontsize=14, color='#f0f6fc',
             fontweight='bold', y=0.98)
plt.tight_layout()
plt.show()


## Key Innovations

### 1. The Degradation Problem
Before ResNet, adding more layers to a network *hurt* accuracy on both training and test sets — not from overfitting, but because optimisation itself became harder. If an identity mapping were optimal, a plain network had no way to express it easily. ResNet solves this by reformulating the target:

Instead of learning the direct mapping H(x), each block learns the **residual** F(x) = H(x) − x.  
Then output = **F(x) + x** (the skip connection adds back x).

If the identity is optimal, F(x) → 0 is easy to learn. This makes optimisation far more tractable.

### 2. Two Block Designs
- **BasicBlock** (ResNet-18/34): Two 3×3 convolutions. Simple, fewer parameters, good for smaller models.
- **Bottleneck** (ResNet-50+): 1×1 squeeze → 3×3 conv → 1×1 expand (expansion=4). Reduces FLOPs since the 3×3 conv operates on *planes* channels, not *planes×4*. This allows deeper networks at similar cost.

### 3. Projection Shortcut
When input and output dimensions differ (either channel count or spatial size), the skip connection uses a 1×1 conv + BN. He et al. found this performs marginally better than zero-padding approaches.

### 4. Zero-Init of Last BN (Goyal et al. 2017)
The last BN in each residual branch is weight-initialized to 0. This makes each block an **identity at the start of training**, which stabilises gradients when training with large batches and improves convergence speed.

### 5. Stage Structure
| Stage | Stride | BasicBlock dim | Bottleneck dim | Output size (224px input) |
|-------|--------|----------------|----------------|---------------------------|
| Stem  | 4×     | 64             | 64             | 56 × 56                   |
| 0     | 1×     | 64             | 256            | 56 × 56                   |
| 1     | 2×     | 128            | 512            | 28 × 28                   |
| 2     | 2×     | 256            | 1024           | 14 × 14                   |
| 3     | 2×     | 512            | 2048           | 7 × 7                     |

### 6. Weight Initialisation
All convolutions use **Kaiming normal** (He init) with `fan_out` and `nonlinearity='relu'`. This correctly calibrates the variance of activations under ReLU, preventing vanishing/exploding gradients at initialisation.

## Component Walkthrough

### `BasicBlock` (lines 64–107 in `resnet.py`)
```python
class BasicBlock(nn.Module):
    expansion: int = 1  # output channels = planes * expansion = planes

    def __init__(self, in_planes, planes, stride=1, downsample=None):
        # First 3×3 conv — stride here performs spatial downsampling if stride>1
        self.conv1 = nn.Conv2d(in_planes, planes, 3, stride=stride, padding=1, bias=False)
        self.bn1   = nn.BatchNorm2d(planes)
        # Second 3×3 conv — always stride=1
        self.conv2 = nn.Conv2d(planes, planes, 3, padding=1, bias=False)
        self.bn2   = nn.BatchNorm2d(planes)
        self.downsample = downsample  # None = identity shortcut

    def forward(self, x):
        identity = x
        out = F.relu(self.bn1(self.conv1(x)), inplace=True)
        out = self.bn2(self.conv2(out))
        if self.downsample is not None:
            identity = self.downsample(x)  # project to match dims
        return F.relu(out + identity, inplace=True)  # RESIDUAL ADD
```

### `Bottleneck` (lines 110–160 in `resnet.py`)
```python
class Bottleneck(nn.Module):
    expansion: int = 4  # output = planes × 4

    def __init__(self, in_planes, planes, stride=1, downsample=None):
        self.conv1 = nn.Conv2d(in_planes, planes, 1, bias=False)           # 1×1 squeeze
        self.conv2 = nn.Conv2d(planes, planes, 3, stride=stride,           # 3×3 main
                                padding=1, bias=False)
        self.conv3 = nn.Conv2d(planes, planes * 4, 1, bias=False)          # 1×1 expand
        # No ReLU on self.bn3 output — applied AFTER residual add

    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)), inplace=True)                # squeeze
        out = F.relu(self.bn2(self.conv2(out)), inplace=True)              # 3×3
        out = self.bn3(self.conv3(out))                                    # expand (no act)
        return F.relu(out + (self.downsample(x) if self.downsample else x))
```

### `ResNet._make_stage` (lines 241–271 in `resnet.py`)
```python
def _make_stage(self, block, planes, n_blocks, stride):
    downsample = None
    out_channels = planes * block.expansion
    # Projection shortcut needed if spatial resolution OR channel count changes
    if stride != 1 or self._in_planes != out_channels:
        downsample = nn.Sequential(
            nn.Conv2d(self._in_planes, out_channels, 1, stride=stride, bias=False),
            nn.BatchNorm2d(out_channels),
        )
    blocks = [block(self._in_planes, planes, stride, downsample)]
    self._in_planes = out_channels
    # Subsequent blocks: same channels, stride=1, no downsample
    for _ in range(1, n_blocks):
        blocks.append(block(self._in_planes, planes))
    return nn.Sequential(*blocks)
```

**Why no bias in conv layers?**  
BatchNorm subtracts the mean and adds a learnable bias (β). A bias in the conv layer before BN would be cancelled out by BN's mean subtraction — so it's redundant and wastes parameters.

**Why Kaiming init?**  
A standard normal init would produce activations whose variance grows or shrinks exponentially with depth through ReLU. Kaiming normal (He init) ensures `Var(output) = Var(input)` through a ReLU layer, maintaining stable gradient magnitudes from input to output.

In [ ]:
import sys
sys.path.insert(0, '../../src')

import torch
from hale_core.nn.vision import build_vision_backbone

x = torch.randn(2, 3, 224, 224)

for name in ['resnet18', 'resnet50', 'resnet101']:
    bb = build_vision_backbone(name, img_size=224)
    bb.eval()
    with torch.no_grad():
        out = bb(x)
    print(f'{name:12s}  pooled={tuple(out.pooled_output.shape)}')
    print(f'  feature_maps: {[tuple(f.shape) for f in out.feature_maps]}')
    print(f'  parameters: {bb.num_parameters():,}')
    print()

# Freeze example
bb50 = build_vision_backbone('resnet50', img_size=224)
bb50.freeze_up_to_stage(1)  # freeze stem + stage0 + stage1
frozen = sum(1 for p in bb50.parameters() if not p.requires_grad)
trainable = sum(1 for p in bb50.parameters() if p.requires_grad)
print(f'After freeze_up_to_stage(1): {frozen} frozen, {trainable} trainable param groups')


## Model Variants

| Model      | Block       | Layers         | embed_dim | Params  |
|------------|-------------|----------------|-----------|----------|
| resnet18   | BasicBlock  | [2, 2, 2, 2]   | 512       | ~11M    |
| resnet34   | BasicBlock  | [3, 4, 6, 3]   | 512       | ~21M    |
| resnet50   | Bottleneck  | [3, 4, 6, 3]   | 2048      | ~25M    |
| resnet101  | Bottleneck  | [3, 4, 23, 3]  | 2048      | ~44M    |
| resnet152  | Bottleneck  | [3, 8, 36, 3]  | 2048      | ~60M    |
| resnet200  | Bottleneck  | [3, 24, 36, 3] | 2048      | ~64M    |

## References

1. **[Primary]** He, K., Zhang, X., Ren, S., & Sun, J. (2016). **Deep Residual Learning for Image Recognition**. *CVPR 2016*. https://arxiv.org/abs/1512.03385

2. Goyal, P., et al. (2017). **Accurate, Large Minibatch SGD: Training ImageNet in 1 Hour**. *arXiv:1706.02677*. — Introduces zero-init of last BN for large-batch training stability.

3. He, K., Zhang, X., Ren, S., & Sun, J. (2016). **Identity Mappings in Deep Residual Networks**. *ECCV 2016*. — ResNet-V2: pre-activation variant (BN-ReLU before conv) for better gradient flow.

4. Xie, S., et al. (2017). **Aggregated Residual Transformations for Deep Neural Networks (ResNeXt)**. *CVPR 2017*. — Extends ResNet with grouped convolutions for better accuracy/FLOPs trade-off.

5. Huang, G., et al. (2017). **Densely Connected Convolutional Networks (DenseNet)**. *CVPR 2017*. — Alternative to ResNet: connect every layer to every subsequent layer, further easing gradient flow.